<a href="https://colab.research.google.com/github/Zikirullo/computer-vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Menu Detector Model

# NOTES
1. transform.Resize(()) - mutates size of an image
2. tranform.Compose([]) - keeps an order of tasks we have to do
3. transforms.ToTensor() - rasmlarni -> raqamlarga o'z gartirib olamiz, yanni 0 ~ 255 float ko'rinishida esa 0.0 ~ 1.0
4. H(height), W(width), C(channel) tensor ko'rinishida esa C, H, W
5. Normalize = pixels = (pixel - mean) / std  

In [ ]:
# ---------------------------
# Import libraries
# ---------------------------
from google.colab import drive
import torch
import torchvision.transforms as transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np
import torch.nn as nn
from torchvision.models import mobilenet_v2
import torch.optim as optim

In [35]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
# Define Dataset path

DATASET_PATH = "/content/drive/MyDrive/food101_dataset"
print("DATASET_PATH", DATASET_PATH)


CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping | class consolidation
    'cheesecake': 'dessert',     # label grouping | class consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}


CLASSES = ["hamburger", "hot_dog", "dessert", "kebab", "pilaf"]
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


DATASET_PATH /content/drive/MyDrive/food101_dataset
5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [37]:
# ----------------------
# Custom Dataset Class
# ----------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path', img_path)
        label = self.labels[idx]
        # print('label', label)
        try:
            image = Image.open(img_path).convert('RGB') # 3 Channel
            image = Image.open(img_path)
            if image.mode == "P" or image.mode == "RGBA":
                image = image.convert("RGBA").convert("RGB")
            else:
                image = image.convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))
        if self.transform:
            image = self.transform(image)
        return image, label

In [38]:
# --------------------------
# Gather and Split Data
# --------------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
3296


In [39]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [40]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=2)

In [41]:
# pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1") # pretrained model | lightweight
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone | model layer freeze
# model.eval()

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)


device cuda


In [43]:
criterion = nn.CrossEntropyLoss() # Loss Function
optimizer = optim.Adam(model.parameters(), lr=0.001) # Optimizer
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick

In [45]:
# --------------
# Training Loop
# --------------

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
  model.train() # train mode
  running_loss = 0.0 # 70% | 30%Loss | 100%
  for images, labels in train_loader: # Forward and Backward(Backpropagation)
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad() # zero the gradient
    outputs = model(images) # Forward Pass | Dog | 5 Classes
    loss = criterion(outputs, labels) # Calculate Loss
    loss.backward()
    optimizer.step() # Adam optimizer
    running_loss += loss.item() # Track Loss

  # Validation
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
      for images, labels in val_loader:
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  val_acc = 100 * correct / total # Calculate Validation Accuracy
  print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%")

  if val_acc > best_accuracy:
      best_accuracy = val_acc
      torch.save(model.state_dict(), '/content/menu_detector.pth')
      print("Saved new best model!")

Epoch [1/10] Loss: 0.5172, Val Accuracy: 82.77%
Saved new best model!
Epoch [2/10] Loss: 0.3065, Val Accuracy: 87.26%
Saved new best model!
Epoch [3/10] Loss: 0.2588, Val Accuracy: 86.89%
Epoch [4/10] Loss: 0.2328, Val Accuracy: 84.83%
Epoch [5/10] Loss: 0.1916, Val Accuracy: 85.68%
Epoch [6/10] Loss: 0.1377, Val Accuracy: 86.89%
Epoch [7/10] Loss: 0.1491, Val Accuracy: 83.25%
Epoch [8/10] Loss: 0.1239, Val Accuracy: 89.32%
Saved new best model!
Epoch [9/10] Loss: 0.0993, Val Accuracy: 88.71%
Epoch [10/10] Loss: 0.1068, Val Accuracy: 82.28%
